In [13]:
import os
import numpy as np
import pandas as pd
import sys
try:
    import torch
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "--quiet"])
    import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

try:
    from datasets import Dataset
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets", "--quiet"])
    from datasets import Dataset
import subprocess
try:
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        DataCollatorWithPadding,
        TrainingArguments,
        Trainer,
    )
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers", "--quiet"])
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        DataCollatorWithPadding,
        TrainingArguments,
        Trainer,
    )

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256

In [14]:
fake_df = pd.read_csv("Fake.csv")
true_df = pd.read_csv("True.csv")

fake_df["label"] = 1   # fake
true_df["label"] = 0   # real

df = pd.concat([fake_df, true_df], ignore_index=True)
df = df[["title", "text", "label"]].fillna("")

df["content"] = (df["title"] + " " + df["text"]).str.strip()


print(df["label"].value_counts())
df.head(2)

label
1    23481
0    21417
Name: count, dtype: int64


,title,text,label,content
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,1,Donald Trump Sends Out Embarrassing New Year’s...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,1,Drunk Bragging Trump Staffer Started Russian C...


In [15]:
df["content"] = (
    df["content"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[df["content"] != ""].copy()

label_counts = df.groupby("content")["label"].nunique()
conflicting_content = label_counts[label_counts > 1].index

n_before = len(df)
n_conflicts = len(conflicting_content)

df = df[~df["content"].isin(conflicting_content)].copy()

df = df.drop_duplicates(subset=["content"]).reset_index(drop=True)

n_after = len(df)
print(f"Rows before cleaning: {n_before}")
print(f"Conflicting content removed: {n_conflicts}")
print(f"Rows after deduplication: {n_after}")
print(f"Rows removed total: {n_before - n_after}")

Rows before cleaning: 44898
Conflicting content removed: 0
Rows after deduplication: 39101
Rows removed total: 5797


In [16]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)

print(len(train_df), len(val_df), len(test_df))

31280 3910 3911


In [17]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[["content", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["content", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["content", "label"]], preserve_index=False)

def tokenize_fn(batch):
    return tokenizer(batch["content"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map: 100%|██████████| 3911/3911 [00:02<00:00, 1656.39 examples/s]


In [18]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [19]:
import sys
print(sys.executable)  # should show ...Python312\python.exe

%pip install --upgrade "accelerate>=1.1.0"
%pip install --upgrade transformers

c:\Users\joeth\AppData\Local\Programs\Python\Python312\python.exe
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import torch, transformers, accelerate, sys
print(sys.executable)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

c:\Users\joeth\AppData\Local\Programs\Python\Python312\python.exe
torch: 2.15.0+cpu
transformers: 5.22.2
accelerate: 1.15.0


In [21]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir="./distilbert-fake-news",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,   # start small; raise to 3-4 if needed
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 16804.10it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
trainer.train()
val_metrics = trainer.evaluate()
val_metrics

c:\Users\joeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000078,0.002697,0.999488,0.998884,1.000000,0.999442
2,0.000041,0.000188,1.000000,1.000000,1.000000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.99it/s]
c:\Users\joeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.02it/s]
c:\Users\joeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.000041,0.000188,2,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 0.00018841442943084985,
 'eval_accuracy': 1.0,
 'eval_precision': 1.0,
 'eval_recall': 1.0,
 'eval_f1': 1.0}

In [23]:
pred = trainer.predict(test_ds)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

print(classification_report(y_true, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

c:\Users\joeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0     0.9995    1.0000    0.9998      2120
           1     1.0000    0.9994    0.9997      1791

    accuracy                         0.9997      3911
   macro avg     0.9998    0.9997    0.9997      3911
weighted avg     0.9997    0.9997    0.9997      3911

Confusion matrix:
 [[2120    0]
 [   1 1790]]


In [24]:
save_dir = "./distilbert-fake-news-best"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to:", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.49it/s]


Saved to: ./distilbert-fake-news-best


In [26]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification


MODEL_DIR = "./distilbert-fake-news-best"
WELFAKE_CSV = "./WELFake_Dataset.csv"
LABEL_COL = "label"
TEXT_COL = "text"
WELFAKE_ONE_IS_FAKE = True
BATCH_SIZE = 32
MAX_LEN = 256


df_ext = pd.read_csv(WELFAKE_CSV).copy()
df_ext = df_ext[[TEXT_COL, LABEL_COL]].dropna()
df_ext[TEXT_COL] = df_ext[TEXT_COL].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df_ext = df_ext[df_ext[TEXT_COL] != ""]

y_true = df_ext[LABEL_COL].astype(int).values
if not WELFAKE_ONE_IS_FAKE:
    y_true = 1 - y_true  # map to your project convention: 1=fake, 0=real

texts = df_ext[TEXT_COL].tolist()


tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()


preds = []
with torch.no_grad():
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(device)
        logits = model(**enc).logits
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())

y_pred = np.array(preds)

# --- Metrics ---
acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
print(f"External Accuracy : {acc:.4f}")
print(f"External Precision: {precision:.4f}")
print(f"External Recall   : {recall:.4f}")
print(f"External F1       : {f1:.4f}")
print("\nClassification report:\n", classification_report(y_true, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 750.63it/s]


External Accuracy : 0.8846
External Precision: 0.8608
External Recall   : 0.9226
External F1       : 0.8906

Classification report:
               precision    recall  f1-score   support

           0     0.9133    0.8452    0.8779     35027
           1     0.8608    0.9226    0.8906     36324

    accuracy                         0.8846     71351
   macro avg     0.8870    0.8839    0.8843     71351
weighted avg     0.8865    0.8846    0.8844     71351

Confusion matrix:
 [[29606  5421]
 [ 2811 33513]]
